# 📘 DATABASE MANAGEMENT AND DESIGN
### *A Practical Learning Series*

# Notebook 3.1 — SQL and Data Analysis — Beginner Level
### SELECT, FROM, WHERE, and ORDER BY

**By Amin Amirkhalili** — Business & Data Analyst

*Based on Chapter 2 (Sections 2-1 & 2-2) and Chapter 6 (Sections 6-2 & 6-3) of:* **Database Management and Design** — Gove Allen, PhD · Gary Hansen, PhD · Robert Jackson, PhD


## 🗺️ Series Roadmap

This notebook is part of a practical learning series on database management and design. The series follows a logical progression — from understanding business needs, through conceptual modelling, all the way to writing SQL code on real databases.

| | |
|---|---|
| **1 — Fundamentals** | |
| Notebook 1 | Introduction: The Database Is the Heart of Information Systems |
| **2 — Database Management** | |
| Notebook 2.1 | Conceptual Data Model: From Conceptual Design to a Relational Model |
| Notebook 2.2 | Database Design: From Schema Creation to Data Management |
| **3 — SQL & Data Analysis** | |
| **Notebook 3.1** | Beginner Level: SELECT, FROM, WHERE, ORDER BY **◀ You are here** |
| Notebook 3.2 | Intermediate Level: Functions, GROUP BY, HAVING |
| Notebook 3.3 | Advanced Level I: Joining Tables — Inner & Outer Joins |
| Notebook 3.4 | Advanced Level II: Subqueries, Views, Temp Tables, CTEs |
| Notebook 3.5 | Data Cleaning with SQL *(coming soon)* |
| **4 — Data Mining** | |
| Notebook 4.1 | SQL in Data Mining *(coming soon)* |

## 📌 What This Notebook Covers

The SQL SELECT statement — selecting columns, filtering rows with WHERE, and sorting results with ORDER BY.

## 💻 How to Use This Notebook

This is the **live practice version** — every query below runs for real.

1. Run the setup cell below **first** (▶ button on the left). It builds a small sample database with the same tables used in the series.
2. Run each query cell as you read. Change the queries — experiment!
3. To start fresh, just run the setup cell again.

> **Note on SQL dialects:** the reading copy of this notebook uses **SQL Server (SSMS)**. Here we run **SQLite** (built into Colab, no installation needed). The two are almost identical; wherever the syntax differs (e.g. `TOP` vs `LIMIT`, `+` vs `||` for text), a note shows both versions.

In [ ]:
#@title ▶️ Run this cell first — it builds the practice database { display-mode: "form" }
# This cell creates a small sample database (SQLite) with the same tables used
# in the notebook: Customer, Manufacturer, Product, Sale, SaleItem, Employee,
# SalaryEmployee, WageEmployee, Purchase, Amin, SimplifiedSales, and more.
# It also defines q("...") which runs a SQL query and shows the result.

import sqlite3, math, datetime
import pandas as pd

conn = sqlite3.connect(":memory:")
conn.executescript("""
PRAGMA foreign_keys = ON;

CREATE TABLE Customer (
  CustomerID INT NOT NULL, FirstName VARCHAR(20), LastName VARCHAR(50),
  StreetAddress VARCHAR(80), City VARCHAR(40), State VARCHAR(2),
  PostalCode VARCHAR(10), Country VARCHAR(30), Phone VARCHAR(20),
  PRIMARY KEY (CustomerID));

CREATE TABLE Manufacturer (
  ManufacturerID INT NOT NULL, ManufacturerName VARCHAR(50),
  Address VARCHAR(80), City VARCHAR(40), State VARCHAR(2),
  PostalCode VARCHAR(10), Phone VARCHAR(20),
  PRIMARY KEY (ManufacturerID));

CREATE TABLE Product (
  ProductID INT NOT NULL, ProductName VARCHAR(60), ManufacturerID INT,
  Category VARCHAR(30), Color VARCHAR(20), Gender VARCHAR(1),
  ListPrice NUMERIC(8,2), Description VARCHAR(120),
  PRIMARY KEY (ProductID),
  FOREIGN KEY (ManufacturerID) REFERENCES Manufacturer(ManufacturerID));

CREATE TABLE Sale (
  SaleID INT NOT NULL, SaleDate DATE NOT NULL,
  Tax NUMERIC(8,2), Shipping NUMERIC(8,2), CustomerID INT,
  PRIMARY KEY (SaleID),
  FOREIGN KEY (CustomerID) REFERENCES Customer(CustomerID));

CREATE TABLE SaleItem (
  SaleID INT NOT NULL, ProductID INT NOT NULL, ItemSize NUMERIC(3,1) NOT NULL,
  Quantity INT NOT NULL, SalePrice NUMERIC(8,2) NOT NULL,
  PRIMARY KEY (SaleID, ProductID, ItemSize),
  FOREIGN KEY (SaleID) REFERENCES Sale(SaleID),
  FOREIGN KEY (ProductID) REFERENCES Product(ProductID));

CREATE TABLE Employee (
  EmployeeID INT NOT NULL, FirstName VARCHAR(20), LastName VARCHAR(50),
  Age INT, City VARCHAR(40), ManagerID INT,
  PRIMARY KEY (EmployeeID),
  FOREIGN KEY (ManagerID) REFERENCES Employee(EmployeeID));

CREATE TABLE SalaryEmployee (
  EmployeeID INT NOT NULL, Salary NUMERIC(10,2),
  PRIMARY KEY (EmployeeID),
  FOREIGN KEY (EmployeeID) REFERENCES Employee(EmployeeID));

CREATE TABLE WageEmployee (
  EmployeeID INT NOT NULL, Wage NUMERIC(8,2), MaxHours INT,
  PRIMARY KEY (EmployeeID),
  FOREIGN KEY (EmployeeID) REFERENCES Employee(EmployeeID));

CREATE TABLE Purchase (
  PurchaseID INT NOT NULL, PurchaseDate DATE, EmployeeID INT, ManufacturerID INT,
  PRIMARY KEY (PurchaseID),
  FOREIGN KEY (EmployeeID) REFERENCES Employee(EmployeeID),
  FOREIGN KEY (ManufacturerID) REFERENCES Manufacturer(ManufacturerID));

CREATE TABLE Amin (
  ID INT NOT NULL, FirstName VARCHAR(20), LastName VARCHAR(50),
  Age INT, Gender VARCHAR(10), PRIMARY KEY (ID));

CREATE TABLE Population (
  Country VARCHAR(40), City VARCHAR(40), Population INT);

CREATE TABLE CovidDeaths (
  location VARCHAR(40), continent VARCHAR(20), date DATE,
  total_cases INT, population INT);

INSERT INTO Manufacturer VALUES
 (1,'Nike','1 Bowerman Dr','Boston','NJ','07101','555-0101'),
 (2,'Adidas','5 Sport Ave','Boston','NJ','07102','555-0102'),
 (3,'Puma','9 Cat St','Chicago','IL','60601','555-0103'),
 (4,'Fila','3 Retro Rd','San Diego','CA','92101','555-0104'),
 (5,'Fashion Lab','7 Style Blvd','Los Angeles','CA','90001','555-0105'),
 (6,'GAP','11 Cotton Way','Concord','NH','03301','555-0106');

INSERT INTO Customer VALUES
 (1,'Sara','Lopez','12 Main St','Boston','VA','23220','USA','555-1001'),
 (2,'Amin','Amirkhalili','34 Oak Ave','Boston','MA','02108','USA','555-1002'),
 (3,'Kim','Nguyen','56 Pine Rd','Chicago','IL','60614','USA','555-1003'),
 (4,'John','Smith','78 Elm St','Richmond','VA','23230','USA','555-1004'),
 (5,'George','Brown','90 Cedar Ln','Springfield','IL','62701','USA','555-1005'),
 (6,'Kevin','Miller','21 Birch Ct','San Diego','CA','92103','USA','555-1006'),
 (7,'Sam','Wilson','43 Maple Dr','Portland','OR','97201','USA','555-1007'),
 (8,'Elli','Rahimi','65 Walnut St','Los Angeles','CA','90012','USA','555-1008');

INSERT INTO Product VALUES
 (1,'AirRun Sneaker',1,'Sneakers','Black','M',120,'Running sneaker'),
 (2,'AirRun Sneaker W',1,'Sneakers','White','F',115,'Running sneaker'),
 (3,'ClassicSandal',1,'Sandals','Brown','F',60,'Summer sandal'),
 (4,'StreetBoot',2,'Boots','Black','M',150,'Urban boot'),
 (5,'TrailBoot',2,'Boots','Brown','F',160,'Hiking boot'),
 (6,'SpeedCat Sneaker',3,'Sneakers','Red','M',95,'Classic sneaker'),
 (7,'BeachSandal',3,'Sandals','Gold','F',45,'Beach sandal'),
 (8,'RetroRunner',4,'Sneakers','Black','F',85,'Retro sneaker'),
 (9,'HeritageBoot',4,'Boots','Tan','M',140,'Heritage boot'),
 (10,'GlamHeel',5,'Heels','Black','F',130,'Evening heel'),
 (11,'CityFlat',5,'Flats','Blue','F',70,'Comfort flat'),
 (12,'CanvasClassic',6,'Sneakers','White','M',55,'Canvas sneaker'),
 (13,'WinterBoot',6,'Boots','Black','F',110,'Insulated boot');

INSERT INTO Sale VALUES
 (1001,'2026-07-03',8.40,5.00,1),
 (1002,'2026-07-02',6.20,5.00,2),
 (1003,'2026-06-28',12.10,0.00,3),
 (1004,'2026-06-15',4.75,5.00,4),
 (1005,'2026-05-30',9.90,7.50,2),
 (1006,'2026-01-18',7.25,5.00,5),
 (1007,'2026-01-05',3.60,0.00,6),
 (1008,'2025-12-20',10.10,5.00,1),
 (1009,'2025-12-08',5.45,5.00,8),
 (1010,'2026-07-10',6.80,0.00,3);

INSERT INTO SaleItem VALUES
 (1001,1,10.0,10,100.00),(1001,4,9.5,5,200.00),
 (1002,6,8.0,1,150.00),
 (1003,2,7.5,2,110.00),(1003,7,6.0,1,42.00),
 (1004,8,7.0,1,80.00),
 (1005,10,6.5,1,125.00),(1005,11,6.0,2,65.00),
 (1006,12,11.0,3,50.00),(1006,1,10.5,1,110.00),
 (1007,9,10.5,1,135.00),
 (1008,13,8.5,2,105.00),(1008,3,7.0,1,55.00),
 (1009,5,8.0,1,155.00),
 (1010,1,9.0,1,118.00);

INSERT INTO Employee VALUES
 (1,'Alan','Stone',55,'Boston',NULL),
 (2,'Ahmad','Karimi',41,'Boston',1),
 (3,'Elli','Moore',38,'Chicago',1),
 (4,'Luke','Perry',29,'San Diego',2),
 (5,'James','Chen',33,'San Diego',2),
 (6,'Alex','Diaz',26,'Portland',3);

INSERT INTO SalaryEmployee VALUES
 (1,95000),(2,72000),(3,68000),(4,80000);

INSERT INTO WageEmployee VALUES
 (5,28.50,40),(6,22.00,30);

INSERT INTO Purchase VALUES
 (501,'2025-12-05',2,1),
 (502,'2025-12-12',4,2),
 (503,'2025-12-27',5,3),
 (504,'2026-02-10',2,4),
 (505,'2026-03-22',3,5),
 (506,'2025-12-30',NULL,6);

INSERT INTO Amin VALUES
 (1,'Amin','Amirkhalili',30,'Male'),
 (2,'Amin','Amirkhalili',50,'Male'),
 (3,'Sara','Lopez',27,'Female'),
 (4,'Elli','Rahimi',35,'Female'),
 (5,'Sadra','Kazemi',22,'Male'),
 (6,'Hossein','Nouri',44,'Male'),
 (7,'Kim','Nguyen',31,'Female');

INSERT INTO Population VALUES
 ('Canada','Toronto',2794356),('Canada','Montreal',1762949),
 ('Canada','Vancouver',662248),('USA','New York',8804190),
 ('USA','Los Angeles',3898747),('USA','Chicago',2746388),
 ('Germany','Berlin',3644826),('Germany','Munich',1471508);

INSERT INTO CovidDeaths VALUES
 ('Canada','North America','2021-06-01',1385000,38000000),
 ('Canada','North America','2021-06-15',1401000,38000000),
 ('Canada','North America','2021-07-01',1412000,38000000),
 ('Germany','Europe','2021-06-01',3700000,83000000),
 ('Germany','Europe','2021-06-15',3717000,83000000),
 ('Germany','Europe','2021-07-01',3730000,83000000),
 ('Iran','Asia','2021-06-01',2950000,85000000),
 ('Iran','Asia','2021-06-15',3020000,85000000),
 ('Iran','Asia','2021-07-01',3230000,85000000);

CREATE TABLE SimplifiedSales AS
SELECT s.SaleID, s.SaleDate, s.CustomerID, p.ProductName, p.Category, p.Color,
       m.ManufacturerName, si.Quantity, si.SalePrice
FROM Sale s
JOIN SaleItem si ON s.SaleID = si.SaleID
JOIN Product p  ON si.ProductID = p.ProductID
JOIN Manufacturer m ON p.ManufacturerID = m.ManufacturerID;
""")

# --- Register SQL-Server-style functions so the book's code runs in SQLite ---
conn.create_function("LEN", 1, lambda s: len(s) if s is not None else None)
conn.create_function("CHARINDEX", 2, lambda sub, s: (s.find(sub) + 1) if (s is not None and sub is not None) else None)
conn.create_function("SQRT", 1, lambda x: math.sqrt(x) if x is not None else None)
conn.create_function("POWER", 2, lambda x, n: x ** n if x is not None else None)
conn.create_function("CEILING", 1, lambda x: math.ceil(x) if x is not None else None)
conn.create_function("FLOOR", 1, lambda x: math.floor(x) if x is not None else None)
conn.create_function("GETDATE", 0, lambda: datetime.date.today().isoformat())
conn.create_function("MONTH", 1, lambda d: int(str(d)[5:7]) if d else None)
conn.create_function("YEAR", 1, lambda d: int(str(d)[0:4]) if d else None)

class _StringAgg:
    def __init__(self): self.items = []
    def step(self, value, sep): self.sep = sep; self.items.append(str(value))
    def finalize(self): return self.sep.join(self.items) if self.items else None
conn.create_aggregate("STRING_AGG", 2, _StringAgg)

def q(sql):
    """Run a SQL query and show the result as a table."""
    sql_stripped = sql.strip().rstrip(";").strip()
    first = sql_stripped.split()[0].upper() if sql_stripped else ""
    if first in ("SELECT", "WITH"):
        return pd.read_sql_query(sql, conn)
    conn.executescript(sql)
    print("OK —", first, "executed.")

print("✅ Practice database is ready. Use q(\"...\") to run SQL. Example:")
q("SELECT * FROM Customer LIMIT 3")


## Intro

The job started when your boss asked:

- Which products sell best? Which products sell the worst? When are sales of these products best?
- Which manufacturers make the most popular products, and which the least popular products?
- Who are our best customers? What kinds of products do they tend to buy? Which brands do they prefer? How often do they purchase: monthly, yearly, twice a year? Do they buy at the beginning of each month, in the middle, or at the end?

To answer these types of questions, we need the SQL **SELECT** statement.

**A SQL SELECT statement is the basic SQL language statement that is used to extract data from a table or a set of tables.**

### The 6 main basic commands

1. SELECT
2. FROM *(+ JOIN, advanced level)*
3. WHERE
4. ORDER BY
5. GROUP BY
6. HAVING

The logic is always like this: you first **SELECT** the columns **FROM** the table (of a database), filter them (**WHERE**) if needed, group the output if needed, and finally order the result by different criteria.

Now let's dive in.

## 1 & 2 — SELECT and FROM

**What I want to do: show me all attributes from a table.**

How do I do it:

```sql
SELECT * FROM Table_Name
```

In [ ]:
q("""
SELECT * FROM Customer
""")

**Show me the top 50 rows** — in SQL Server: `SELECT TOP 50 * FROM Customer`.
In SQLite the same thing is written with `LIMIT`:

In [ ]:
q("""
SELECT * FROM Customer LIMIT 50
""")

**What I want to do: show me some attributes from a table.**

```sql
SELECT Attribute_Names FROM Table_Name
```

In [ ]:
q("""
SELECT FirstName, LastName FROM Customer
""")

I can name the columns however I like with `AS`:

In [ ]:
q("""
SELECT FirstName AS Name_I_Like FROM Customer
""")

### DISTINCT

`SELECT DISTINCT attribute_names FROM table_name` — it only shows unique data for each attribute.

In [ ]:
q("""
SELECT DISTINCT City FROM Customer
""")

Note that the DISTINCT keyword does not eliminate duplicate values from individual columns, but rather duplicate **rows** from the query result.

It means that if the table contains two people called Amin Amirkhalili with different ages, and you say:

In [ ]:
q("""
SELECT DISTINCT FirstName, LastName, Age FROM Amin
""")

it shows both rows — because rows are compared as a whole, not only by name or last name.

## 3 — WHERE

We limit the result by adding more specifications.

**What I want to do: show me all attributes from a table that have this speciality.**

```sql
SELECT * FROM Table_Name
WHERE [a condition that is True/False]
```

### Using `=` or `LIKE` for string variables

In [ ]:
q("""
SELECT * FROM Customer
WHERE State = 'VA'
""")

In [ ]:
q("""
SELECT * FROM Customer
WHERE State LIKE 'VA'
""")

### Using `=`, `>`, `<`, `<>`, `<=`, `>=` for numbers

In [ ]:
q("""
SELECT * FROM Product
WHERE ListPrice = 150
""")

### Adding AND, OR, AND NOT, OR NOT to add limitations

Using parentheses is important and can change the query result.

In [ ]:
q("""
SELECT * FROM SimplifiedSales
WHERE Category = 'Sandals'
  AND Color <> 'Gold'
""")

Example — **with** parentheses:

In [ ]:
q("""
SELECT * FROM SimplifiedSales
WHERE (Category = 'Sandals' OR Category = 'Sneakers')
  AND ManufacturerName = 'Nike'
""")

Now the same query **without** parentheses — compare the results! Without parentheses, `AND` binds before `OR`, so all sandals are returned, from any manufacturer:

In [ ]:
q("""
SELECT * FROM SimplifiedSales
WHERE Category = 'Sandals' OR Category = 'Sneakers'
  AND ManufacturerName = 'Nike'
""")

**Query: show sales of boots not made by either Nike or Puma.**

In [ ]:
q("""
SELECT *
FROM SimplifiedSales
WHERE Category = 'Boots'
  AND NOT (ManufacturerName = 'Nike'
        OR ManufacturerName = 'Puma')
""")

## 4 — ORDER BY

This command orders the result.

**What I want to do: order the result for me based on the condition I tell you.**

```sql
SELECT * FROM Table_Name
ORDER BY Attribute [ASC | DESC]
```

- **ASC**: from small to large (or from A to Z). This is the default.
- **DESC**: from large to small (or from Z to A).

In [ ]:
q("""
SELECT * FROM SaleItem
ORDER BY SalePrice DESC
""")

Remember: ORDER BY usually comes at the end (here, meaning after SELECT and WHERE).

In [ ]:
q("""
SELECT DISTINCT
    Category,
    Color,
    ManufacturerName
FROM SimplifiedSales
WHERE Category = 'Boots'
ORDER BY ManufacturerName
""")

Note that ORDER BY can use two or more attributes to order by:

In [ ]:
q("""
SELECT DISTINCT
    Category,
    Color,
    ManufacturerName
FROM SimplifiedSales
WHERE Category = 'Boots'
ORDER BY ManufacturerName, Color DESC
""")

---
### 🎯 Your turn — practice ideas

1. List all products cheaper than $100, ordered by price.
2. Show the distinct manufacturer names in `SimplifiedSales`.
3. Find all sales of black products that are **not** sneakers.

---
⏭️ **Coming up in Notebook 3.2:** functions in SELECT, GROUP BY, and HAVING.